# 像素级 H-VO2 视频处理流程

本 Notebook 按功能模块展示视频读取、亮度到电流映射、逐帧阶梯输入、像素阵列仿真、完成度统计，以及结果图保存和目录命名。

为避免 Notebook 与 `video_process.py` 的逻辑分叉，下面各模块直接复用该脚本中的实现；最后一格通过公开接口运行完整流程。

In [1]:
from pathlib import Path

from video_process import (
    DEFAULT_FPS,
    DEFAULT_SIMULATION_DT_S,
    STOP_WINDOW_STEPS,
    PixelReactivationError,
    VideoPixelProcessor,
    process_video_file,
)

# 支持从工作区根目录或 neuron_model 目录运行 Notebook。
_candidates = (
    Path.cwd(),
    Path.cwd() / "HVO2" / "neuron_model",
    Path.cwd() / "neuron_model",
)
PROJECT_DIR = next(
    (path for path in _candidates if (path / "Weizmann_data").is_dir()),
    Path.cwd(),
)
VIDEO_PATH = PROJECT_DIR / "Weizmann_data" / "shahar_side.avi"
OUTPUT_ROOT = PROJECT_DIR / "outputs"

FPS = 50.0
CURRENT_MIN_MA = 2.0
CURRENT_MAX_MA = 12.0
GAIN = 1.0
SIMULATION_DT_S = DEFAULT_SIMULATION_DT_S
COMPLETION_DIVISIONS = 5
METRICS = ("pulse_frequency",)
START_FRAME = 0
TITLE = "shahar_side_notebook"

processor = VideoPixelProcessor(
    fps=FPS,
    current_min_ma=CURRENT_MIN_MA,
    current_max_ma=CURRENT_MAX_MA,
    gain=GAIN,
    simulation_dt_s=SIMULATION_DT_S,
)

print(f"视频: {VIDEO_PATH}")
print(f"仿真步长: {SIMULATION_DT_S * 1000:g} ms；停振观察: {STOP_WINDOW_STEPS} 步")

视频: e:\vscode\HVO2\neuron_model\Weizmann_data\shahar_side.avi
仿真步长: 1 ms；停振观察: 3 步


In [2]:
import cv2
import numpy as np

## 1. 视频读取模块

与 `VideoPixelProcessor.read_video_grayscale` 一致：从指定起始帧读取视频，将 BGR 通道以浮点数加权为亮度，返回 `(帧数, 高, 宽)` 的 `float32` 数组。

In [4]:
def read_video_grayscale(video_path: str | Path, start_frame: int = 0) -> np.ndarray:
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"video file not found: {video_path}")

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"failed to open video: {video_path}")

    frames = []
    frame_index = 0
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        if frame_index >= start_frame:
            bgr = frame.astype(np.float32)
            luminance = (
                0.114 * bgr[:, :, 0]
                + 0.587 * bgr[:, :, 1]
                + 0.299 * bgr[:, :, 2]
            )
            frames.append(luminance)
        frame_index += 1
    capture.release()

    if not frames:
        raise ValueError(f"no frames available in video: {video_path}")
    return np.stack(frames, axis=0)

# 执行读取，后续模块使用同一份中间数据。
gray_frames = read_video_grayscale(VIDEO_PATH, start_frame=START_FRAME)
print(f"帧数组: shape={gray_frames.shape}, dtype={gray_frames.dtype}")

帧数组: shape=(43, 144, 180), dtype=float32


## 2. 像素映射模块：亮度 → 光强 → 电流

浮点亮度先除以 255 得到 0–1 归一化光强，再按当前脚本的线性公式映射到 `current_min_ma`–`current_max_ma`，最后乘以 `gain`。

In [5]:
def pixel_to_current(
    gray_frames: np.ndarray,
    *,
    current_min_ma: float,
    current_max_ma: float,
    gain: float,
) -> np.ndarray:
    gray_frames = np.asarray(gray_frames, dtype=np.float32)
    if gray_frames.ndim != 3:
        raise ValueError("gray_frames must be 3D with shape (frames, height, width)")

    normalized = gray_frames / 255.0
    current = current_min_ma + (current_max_ma - current_min_ma) * normalized
    return current * gain

current_array = pixel_to_current(
    gray_frames,
    current_min_ma=CURRENT_MIN_MA,
    current_max_ma=CURRENT_MAX_MA,
    gain=GAIN,
)
print(f"电流阵列: shape={current_array.shape}, min={current_array.min():.4f} mA, max={current_array.max():.4f} mA")

电流阵列: shape=(43, 144, 180), min=2.0235 mA, max=12.0000 mA


## 3. 阶梯电流序列生成模块

每帧电流在该帧时间区间内保持不变；该区间长度为 `1 / FPS`。仿真内部步长可以更小，函数会按时间取对应视频帧的电流。

In [6]:
def make_step_current_function(
    frame_currents: np.ndarray,
    *,
    fps: float,
) -> tuple[float, object]:
    frame_dt_s = 1.0 / fps
    total_duration_s = float(frame_currents.size) * frame_dt_s

    def current_function(t_s: float) -> float:
        if t_s < 0.0:
            return float(frame_currents[0])
        if t_s >= total_duration_s:
            return 0.0
        index = int(np.floor(t_s / frame_dt_s))
        index = min(max(index, 0), frame_currents.size - 1)
        return float(frame_currents[index])

    return total_duration_s, current_function

# 查看一个示例像素的阶梯电流函数（完整仿真会逐像素创建它）。
example_y, example_x = current_array.shape[1] // 2, current_array.shape[2] // 2
example_duration_s, example_current = make_step_current_function(
    current_array[:, example_y, example_x], fps=FPS
)
print(f"示例像素阶梯信号时长: {example_duration_s:.3f} s")
print(f"前 3 帧电流: {[example_current(i / FPS) for i in range(min(3, len(gray_frames)))]}")

示例像素阶梯信号时长: 0.860 s
前 3 帧电流: [6.953216075897217, 7.004157066345215, 6.953216075897217]


## 4. 模型阵列仿真模块

每个像素序列交给 `VideoPixelProcessor._simulate_single_pixel`，因此复用源脚本的 1 ms（可配置）积分、固定 3 步复振检查、理论寿命上界和统计规则。这里先用单个像素验证接口；完整阵列在最后的运行单元执行。

In [7]:
def simulate_pixel_array(
    processor: VideoPixelProcessor,
    current_array: np.ndarray,
) -> dict[str, np.ndarray]:
    height, width = current_array.shape[1:]
    response_map = np.full((height, width), np.nan, dtype=float)
    pulse_count_map = np.full((height, width), np.nan, dtype=float)
    pulse_frequency_map = np.full((height, width), np.nan, dtype=float)
    completion_time_map = np.full((height, width), np.nan, dtype=float)
    reactivated_pixels = []

    for y in range(height):
        for x in range(width):
            pixel_result, reactivated, reactivation_indices = processor._simulate_single_pixel(
                current_array[:, y, x],
                pixel_xy=(y, x),
            )
            if reactivated:
                reactivated_pixels.extend(reactivation_indices)
                continue
            if pixel_result is None:
                continue

            response_map[y, x] = pixel_result.response_duration_s
            pulse_count_map[y, x] = float(pixel_result.pulse_count)
            pulse_frequency_map[y, x] = pixel_result.pulse_frequency_hz
            completion_time_map[y, x] = pixel_result.completion_time_s

    if reactivated_pixels:
        unique_pixels = sorted(set(reactivated_pixels))
        raise PixelReactivationError(unique_pixels, len(unique_pixels))

    return {
        "duration": response_map,
        "pulse_count": pulse_count_map,
        "pulse_frequency": pulse_frequency_map,
        "completion_time": completion_time_map,
    }

# 用示例像素检查模型阵列的单像素接口，不在此处重复运行整幅图像。
example_result, example_reactivated, _ = processor._simulate_single_pixel(
    current_array[:, example_y, example_x],
    pixel_xy=(example_y, example_x),
)
print("示例像素结果:", example_result)
print("复振异常:", example_reactivated)

示例像素结果: PixelResult(response_duration_s=0.106, pulse_count=2261, pulse_frequency_hz=21337.29656654573, completion_time_s=0.109)
复振异常: False


## 5. 结果统计模块

已完成像素按 `completion_time` 升序排序后取前 N 等分；每个分组独立生成 mask，最后一组强制选入全部已完成像素。指标名称及校验与公开接口保持一致。

In [3]:
from typing import Iterable

In [8]:
def normalize_metrics(metrics: Iterable[str]) -> tuple[str, ...]:
    if isinstance(metrics, str):
        selected = (metrics,)
    else:
        selected = tuple(dict.fromkeys(metrics))
    available = {"duration", "pulse_frequency", "pulse_count"}
    if not selected:
        raise ValueError("metrics must contain at least one metric")
    unknown = set(selected) - available
    if unknown:
        raise ValueError(
            f"unsupported metrics: {', '.join(sorted(unknown))}; "
            f"choose from {', '.join(sorted(available))}"
        )
    return selected


def select_completion_groups(
    completion_time_map: np.ndarray,
    completion_divisions: int,
) -> list[tuple[int, np.ndarray]]:
    if not isinstance(completion_divisions, int) or isinstance(completion_divisions, bool):
        raise TypeError("completion_divisions must be an integer")
    if not 1 <= completion_divisions <= 100:
        raise ValueError("completion_divisions must be between 1 and 100")

    completed_mask = np.isfinite(completion_time_map)
    completed_flat_indices = np.flatnonzero(completed_mask)
    if completed_flat_indices.size == 0:
        raise RuntimeError("no pixels completed the first stop criterion; cannot build output maps")

    flat_times = completion_time_map.ravel()[completed_flat_indices]
    sorted_order = np.argsort(flat_times, kind="mergesort")
    total = completed_flat_indices.size
    groups = []

    for division_index in range(1, completion_divisions + 1):
        percentage = division_index * 100 // completion_divisions
        if division_index == completion_divisions:
            target_count = total
        else:
            target_count = int(np.round(total * division_index / completion_divisions))
            target_count = max(1, min(target_count, total))

        selected_flat = completed_flat_indices[sorted_order[:target_count]]
        selected_y, selected_x = np.unravel_index(selected_flat, completion_time_map.shape)
        mask = np.zeros_like(completed_mask, dtype=bool)
        mask[selected_y, selected_x] = True
        groups.append((percentage, mask))

    return groups

selected_metrics = normalize_metrics(METRICS)
print("选择的输出指标:", selected_metrics)

选择的输出指标: ('pulse_frequency',)


## 6. 图像保存模块 与 文件命名模块

输出目录遵循 `MM-DD-HH-MM-title-startN`。每个完成度下只写入所选指标，mask 外像素为黑色；有效像素使用该图当前有效值范围归一化到 0–255。

In [9]:
from datetime import datetime


def make_output_directory(
    output_root: str | Path,
    title: str,
    start_frame: int,
) -> Path:
    timestamp = datetime.now().strftime("%m-%d-%H-%M")
    safe_title = str(title).strip().replace(" ", "_").replace("/", "-")
    output_dir = Path(output_root) / f"{timestamp}-{safe_title}-start{start_frame}"
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def write_metric_images(
    output_dir: Path,
    percentage: int,
    mask: np.ndarray,
    metric_values: dict[str, np.ndarray],
) -> None:
    for metric_name, values in metric_values.items():
        image = np.zeros(values.shape, dtype=np.uint8)
        valid_mask = mask & np.isfinite(values)
        if np.any(valid_mask):
            valid_values = values[valid_mask]
            min_value = np.min(valid_values)
            max_value = np.max(valid_values)
            if np.isclose(max_value, min_value):
                normalized = np.full_like(valid_values, 255.0, dtype=np.float32)
            else:
                normalized = (valid_values - min_value) / (max_value - min_value)
                normalized = normalized * 255.0
            image_values = np.zeros_like(values, dtype=np.float32)
            image_values[valid_mask] = normalized
            image = np.clip(image_values, 0.0, 255.0).astype(np.uint8)

        filename = f"{percentage}%-{metric_name}.png"
        cv2.imwrite(str(output_dir / filename), cv2.cvtColor(image, cv2.COLOR_GRAY2BGR))

print("输出命名和保存函数已定义。")

输出命名和保存函数已定义。


## 运行完整阵列流程

依次执行逐像素仿真、完成度分组、所选指标过滤和图像保存。配置项可在前面的初始化单元修改。

In [10]:
metric_maps = simulate_pixel_array(processor, current_array)
completion_groups = select_completion_groups(
    metric_maps["completion_time"],
    completion_divisions=COMPLETION_DIVISIONS,
)
selected_metric_values = {
    name: metric_maps[name]
    for name in selected_metrics
}
output_dir = make_output_directory(OUTPUT_ROOT, TITLE, START_FRAME)

for percentage, mask in completion_groups:
    write_metric_images(
        output_dir=output_dir,
        percentage=percentage,
        mask=mask,
        metric_values=selected_metric_values,
    )

print(f"输出目录: {output_dir}")
print(f"完成度标签: {[percentage for percentage, _ in completion_groups]}")
print(f"图像文件: {sorted(path.name for path in output_dir.glob('*.png'))}")

输出目录: e:\vscode\HVO2\neuron_model\outputs\10-16-23-43-shahar_side_notebook-start0
完成度标签: [20, 40, 60, 80, 100]
图像文件: ['100%-pulse_frequency.png', '20%-pulse_frequency.png', '40%-pulse_frequency.png', '60%-pulse_frequency.png', '80%-pulse_frequency.png']
